In [4]:
from kinetic_model.constants.energy_profile import EnergyBarriersSimplifyed, EnergyBarriers
from kinetic_model.simulation.run_reactor import ReactorSimulationChemisorptionSimplified, ReactorSimulationChemisorption
from kinetic_model.core.diffusion import Reaction_Diffusion
import seaborn as sns
import joblib
import numpy as np
import pandas as pd
import math
from copy import deepcopy
import re

from kinetic_model.simulation.reaction_list_creator import (
    make_reaction_list_chemisorp_simplified,
    prepare_constants_chemisorp_simplified
)


class ReactorStream(ReactorSimulationChemisorptionSimplified):
    """
    A class to handle the reactor stream simulation.
    Inherits from ReactorSimulationChemisorption.
    """
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def _build_solver(self):
        df, idx = self._make_dataframe()
        self._diffusion_rates = self._calc_diffusion_constants(df, idx)
        self._capacity = self._calc_capacity(df, idx)

        for key, _ in self._capacity.items():
            if 'out' in key:
                self._capacity[key] = 1e6


        self.consts = prepare_constants_chemisorp_simplified(
            self.barrier_params,
            temperature=self.T
        )
        self._reactions = make_reaction_list_chemisorp_simplified(
            n_max=self.N,
            consts=self.consts,
            diffusion_rates=self._diffusion_rates,
            include_diffusion=self.include_diffusion

        )

        for i, reaction in enumerate(self._reactions):
            string = reaction[0].replace('<D>', 'D>')
            self._reactions[i] = (string, reaction[1], reaction[2], )

        return Reaction_Diffusion(reactions=self._reactions, capacity=self._capacity, 
                                  fixed_concentrations=['C2', 'C2_out'])


def time_distribution(result: pd.DataFrame) -> pd.DataFrame:
    """
    Compute per-time distribution P(C_n | n>2) from a wide time series.

    Input
    -----
    result : DataFrame with columns like:
        time, C2, C2_out, C4, C4_out, C6, C6_out, ...

    Output
    ------
    DataFrame with columns:
        time, n, probability   (only n > 2)

    Notes
    -----
    - Uses only in-reactor columns matching exactly 'C\\d+' (excludes '*_out').
    - If total concentration of n>2 at a time is 0, probabilities are NaN for that time.
    """
    if 'time' not in result.columns:
        raise ValueError("Input must contain a 'time' column.")

    # Keep only exact Cn columns (exclude *_out etc.)
    cn_cols = [c for c in result.columns if re.fullmatch(r'C\d+', c)]

    if not cn_cols:
        return pd.DataFrame(columns=['time', 'n', 'probability'])

    # Wide -> long
    long = (result[['time'] + cn_cols]
            .melt(id_vars='time', var_name='species', value_name='concentration'))

    # Extract n
    long['n'] = long['species'].str.extract(r'(\d+)').astype(int)

    # Totals per time over n>2
    totals = (long.loc[long['n'] > 2]
                    .groupby('time', as_index=False)['concentration']
                    .sum()
                    .rename(columns={'concentration': 'total_conc'}))

    # Merge totals, compute probabilities, filter n>2
    out = (long.merge(totals, on='time', how='left')
                .assign(probability=lambda d: d['concentration'] / d['total_conc'])
                .loc[lambda d: d['n'] > 2, ['time', 'n', 'probability']]
                .sort_values(['time', 'n'])
                .reset_index(drop=True))

    return out


def tail_slope_prob_per_n(dist: pd.DataFrame, tail_frac: float = 0.10) -> pd.DataFrame:
    """
    Compute d(probability)/dt for each n over the last `tail_frac` of the simulation time.

    Parameters
    ----------
    dist : pd.DataFrame
        Long-form dataframe with columns: ['time', 'n', 'probability'].
    tail_frac : float, optional (default=0.10)
        Fraction of the *time span* to use from the end (e.g., 0.10 = last 10%).

    Returns
    -------
    pd.DataFrame with columns:
        - n : int
        - slope : float  (units: probability per time)
        - window_start : float  (time at which the tail window starts)
        - window_end : float    (max time)
        - points_used : int     (# of points used in the fit for that n)

    Notes
    -----
    - Uses np.polyfit on (time, probability) within the tail window, after dropping NaNs.
    - If fewer than 2 points for an n in the tail window, slope is NaN.
    """
    required_cols = {'time', 'n', 'probability'}
    missing = required_cols - set(dist.columns)
    if missing:
        raise ValueError(f"Input distribution is missing columns: {missing}")

    # Ensure proper dtypes
    d = dist[['time', 'n', 'probability']].copy()
    d['time'] = pd.to_numeric(d['time'], errors='coerce')
    d['n'] = pd.to_numeric(d['n'], errors='coerce', downcast='integer')
    d['probability'] = pd.to_numeric(d['probability'], errors='coerce')

    # Determine tail time window using the overall time span
    t_min = d['time'].min()
    t_max = d['time'].max()
    if not np.isfinite(t_min) or not np.isfinite(t_max):
        return pd.DataFrame(columns=['n', 'slope', 'window_start', 'window_end', 'points_used'])

    window_start = t_min + (1 - tail_frac) * (t_max - t_min)
    tail = d[(d['time'] >= window_start) & (d['time'] <= t_max)]

    results = []
    for n_val, g in tail.groupby('n', dropna=True):
        g = g.dropna(subset=['time', 'probability']).sort_values('time')
        if len(g) < 2:
            slope = np.nan
            points_used = len(g)
        else:
            # Linear fit: probability ~ a*time + b; slope = a
            a, b = np.polyfit(g['time'].values, g['probability'].values, deg=1)
            slope = float(a)
            points_used = len(g)

        results.append({
            'n': int(n_val) if pd.notna(n_val) else np.nan,
            'slope': slope,
            'window_start': float(window_start),
            'window_end': float(t_max),
            'points_used': int(points_used),
        })

    return pd.DataFrame(results).sort_values('n').reset_index(drop=True)


def check_convergance(ts, error_threshold=1e-2):
    distr = time_distribution(ts.reset_index(names=['time']))
    is_converged = bool(tail_slope_prob_per_n(distr, tail_frac=0.10).slope.abs().max() < error_threshold)
    if is_converged:
        return True
    return False


In [ ]:
N = 8
temperature = 600  # Kelvin
pressure = 1  # bar
steps = 1e4
time = 1e5
diffusion = False  # Set to False to disable diffusion

c2_model = joblib.load('../data/adsorption/pt_poly_model.pkl')

C2_conc, C2_out_conc = c2_model.predict_absolute(np.array([pressure * 101325]), 
                                                 np.array([temperature])), \
                    c2_model.predict_out(np.array([pressure * 101325]), 
                                         np.array([temperature]))

energy_barriers = EnergyBarriersSimplifyed(
    temperature=temperature,
    pressure=pressure
)

reactor = ReactorSimulationChemisorptionSimplified(
    N=N,
    T=temperature,
    pressure=pressure,
    L=1000,
    time_hours=time,
    n_steps=int(steps),
    diff_model_path='../data/processed/diffusion_extrap_place_holder_model.pkl',
    ads_model_path='../data/processed/knn_adsorption_slope_model.pkl',
    energy_model=energy_barriers,
    initial_concs={'CatC2(C2)': 1e-3, 'C2': C2_conc[0], 'C2_out': C2_out_conc[0]},
    include_diffusion=diffusion
)


reactor.solver = reactor._build_solver()

def max_constant(reactions):
    return np.max([i[1:] for i in reactions])

def normilize_constants(reactions, r_max=1e6):
    factor = r_max/max_constant(reactions)
    if factor >=1:
        factor = 1
    for i in range(len(reactions)):
        reactions[i] = (reactions[i][0], reactions[i][1] * factor, reactions[i][2] * factor)
    return factor

def remove_back_ins(reactions, N):
    number_of_insertions = N//2 - 2
    for cnt, i in enumerate(reactions):
        if cnt > number_of_insertions:
            break
        reactions[cnt] = (i[0].replace('<->', '->'), i[1], i[2])

def slowdawn_back_ins(reactions, N, order):
    number_of_insertions = N//2 - 2
    for cnt, i in enumerate(reactions):
        if cnt > number_of_insertions:
            break
        reactions[cnt] = (i[0], i[1], i[2]*10**(-order))

def check_errors(ts):
    return all(ts.min() > -1e-2)

factor = normilize_constants(reactor._reactions, r_max=1e6)
# remove_back_ins(reactor._reactions, reactor.N)
# slowdawn_back_ins(reactor._reactions, reactor.N, order=8)
reactor.time_hours /= factor

for i in reactor._reactions:
    print('{:35s} {:5.2e} {:5.2e}'.format(*i))

distr, ts = reactor.run(build_solver=False)

sns.barplot(data=distr, x='n', y='probability')
check_convergance(ts, error_threshold=1e-8)

In [11]:
def get_energies(temperature, pressure):
    energy_barriers = EnergyBarriersSimplifyed(
        temperature=temperature,
        pressure=pressure
    )

    barriers = energy_barriers.get_energy_barriers()
    
    energies = [0, 
                barriers['deelimination_di'] - barriers['elimination_di'] \
                + barriers['insertion_tri'] - barriers['deinsertion_tri'] \
                + barriers['elimination_tri'] - barriers['deelimination_tri'] \
                ]

    energies = energies + [
                            
            energies[1]    + barriers['deelimination_tri'] - barriers['elimination_tri'] \
                + barriers['insertion_tetra'] - barriers['deinsertion_tetra'] \
                + barriers['elimination_tetra'] - barriers['deelimination_tetra']
    ]

    return energies

get_energies(300, 5)

[0, -51.71004429070591, -103.1749584700734]

In [18]:
def state_probabilities(energies_kjmol, temperature):
    """
    Calculate probabilities of each state using the Boltzmann distribution.

    Parameters
    ----------
    energies_kjmol : array-like
        Energies of states in kJ/mol.
    temperature : float
        Temperature in Kelvin.

    Returns
    -------
    list of float
        Probabilities of each state (sum to 1).
    """
    R = 8.314462618  # J/(mol·K)
    energies_jmol = np.array(energies_kjmol) * 1e3  # convert kJ/mol to J/mol
    boltzmann_factors = np.exp(-energies_jmol / (R * temperature))
    Z = np.sum(boltzmann_factors)
    probabilities = boltzmann_factors / Z
    return probabilities.tolist()


def concentrations(energies_kjmol, temperature):
    probabilities = state_probabilities(energies_kjmol, temperature)
    for cnt, i in enumerate(probabilities):
        probabilities[cnt] = i / ((cnt + 2))
    probabilities = np.array(probabilities)
    probabilities /= probabilities.sum()
    return probabilities

temperature = 600
pressure = 1
energies = get_energies(temperature, pressure)
state_probabilities(energies, temperature), concentrations(energies, temperature)

([0.25455121582358686, 0.32374847420257447, 0.4217003099738387],
 array([0.37366211, 0.31682567, 0.30951223]))